# 6. Reti neurali

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FedeGambe/Master_s_thesis_Data_science/blob/revisione-progetto/notebooks/06_reti_neurali.ipynb)

Reti Keras addestrate sullo stesso split dei modelli di machine learning, compresa la rete avanzata della tesi (189 feature polinomiali, LeakyReLU, L1 + L2, BatchNormalization, AdamW). Differenza chiave rispetto alla tesi: l'early stopping usa un set di **validazione** estratto dal training; nella versione originale usava il test set, rendendo ottimistica la stima finale. L'addestramento delle tre reti richiede circa 2 minuti.

In [ ]:
# Configurazione dell'ambiente (locale o Google Colab)
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("Master_s_thesis_Data_science"):
        !git clone -q -b revisione-progetto https://github.com/FedeGambe/Master_s_thesis_Data_science.git
    %cd Master_s_thesis_Data_science
    !pip install -q -r requirements-colab.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import plotly.io as pio
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)
if os.environ.get("TESI_GRAFICI_STATICI"):
    pio.renderers.default = "png"  # usato solo per salvare output visibili su GitHub
    pio.defaults.default_width = 1000

In [ ]:
from tesi_bev import dati, modelli as M, valutazione as V, grafici as G
from tesi_bev.reti_neurali import ARCHITETTURE, ReteNeurale
X, y = dati.prepara_ml()
X_tr, X_te, y_tr, y_te = M.dividi(X, y)
pd.DataFrame(ARCHITETTURE).T

In [ ]:
reti, prob = {}, {}
for nome in ARCHITETTURE:
    reti[nome] = ReteNeurale(nome).fit(X_tr, y_tr)
    prob[nome] = reti[nome].predict_proba(X_te)[:, 1]
    G.storia_addestramento(reti[nome].storia_, f"{nome}: {len(reti[nome].storia_['loss'])} epoche").show()

In [ ]:
risultati = V.tabella_risultati(prob, y_te).join(V.tabella_ic(prob, y_te, n=500))
risultati[["accuracy", "f1", "roc_auc", "roc_auc_ic_basso", "roc_auc_ic_alto", "brier"]]

## Confronto con i modelli ad alberi

In [ ]:
from pathlib import Path
if Path("risultati/machine_learning/risultati_test.csv").exists():
    ml = pd.read_csv("risultati/machine_learning/risultati_test.csv", index_col=0)
    display(pd.concat([ml.head(3), risultati])[["accuracy", "f1", "roc_auc", "roc_auc_ic_basso", "roc_auc_ic_alto"]])

Su dati tabellari di questa dimensione le reti neurali non superano i modelli a gradient boosting. La rete avanzata,
con quasi 30 volte i parametri della rete regolarizzata, non migliora la ROC-AUC: le feature polinomiali aumentano la
complessità senza aggiungere informazione.